# BM25 Retrieval: Mastering Keyword-Based Document Search

Welcome to the dedicated section on advanced retrieval techniques. While modern RAG systems often default to dense vector embeddings (like those from OpenAI or Cohere), relying solely on semantic similarity can introduce blind spots—especially when dealing with highly technical jargon, specific proper nouns, or exact numerical data. This notebook introduces BM25, a powerful and historically significant keyword-based retrieval method.

BM25 operates fundamentally differently from embedding models. Instead of mapping documents and queries into a continuous vector space to measure semantic distance, it calculates relevance based on classical information retrieval principles: Term Frequency (TF) and Inverse Document Frequency (IDF). It scores how often a query term appears in a document relative to how rare that term is across the entire corpus. This makes BM25 exceptionally robust for tasks requiring precise keyword matching—such as searching legal statutes, technical manuals, or product codes—where the exact presence of a word is more critical than its surrounding context.

Understanding when and why to use BM25 is crucial for building resilient RAG pipelines. In advanced architectures like LangGraph, you might implement a hybrid retrieval step: first using BM25 for high-precision keyword hits (e.g., finding specific product IDs), and then falling back to vector search for broader semantic context (e.g., understanding the general concept of "product ID"). By mastering this technique, you gain the ability to design sophisticated, multi-stage retrieval agents that combine the precision of classical NLP with the breadth of modern deep learning models.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Differentiate Retrieval Methods:** Articulate the core conceptual differences between embedding-based (semantic) and keyword-based (lexical) retrievers.
*   **Implement BM25:** Initialize and utilize `BM25Retriever` within a LangChain workflow without requiring an external vector store or embedding model.
*   **Analyze Strengths and Weaknesses:** Identify scenarios where BM25 excels (e.g., technical jargon, exact names) and scenarios where it fails (e.g., paraphrasing, conceptual similarity).
*   **Design Hybrid RAG Systems:** Understand how to integrate keyword retrieval into a larger, multi-stage Retrieval Augmented Generation (RAG) architecture for improved robustness.


### BM25 Retriever Setup

This cell imports the necessary components to utilize a BM25 retriever. Unlike embedding-based methods, BM25 is a keyword matching technique that scores document relevance based on term frequency and inverse document frequency (TF-IDF), making it useful when vectorization or embeddings are undesirable.


In [8]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# BM25 is a keyword-based retriever — no embeddings or vector store required
# It scores documents by term frequency and inverse document frequency (TF-IDF variant)


### Data Preparation: Loading Documents

This cell initializes a list of `Document` objects. Each document contains specific text content (`page_content`) and associated metadata (like the `topic`). This structured dataset simulates a knowledge base, which is essential for any Retrieval-Augmented Generation (RAG) system to function.


In [9]:
# 12 documents spanning medicine, architecture, finance, and literature
docs = [
    Document(page_content="Antibiotics inhibit bacterial cell wall synthesis or protein production to stop infection.", metadata={"topic": "medicine"}),
    Document(page_content="Vaccines introduce antigens to train the immune system to recognise and neutralise pathogens.", metadata={"topic": "medicine"}),
    Document(page_content="MRI scanners use magnetic fields and radio waves to produce detailed images of soft tissue.", metadata={"topic": "medicine"}),
    Document(page_content="Blood pressure is measured in millimetres of mercury (mmHg) and expressed as systolic over diastolic.", metadata={"topic": "medicine"}),
    Document(page_content="The Pantheon in Rome was built around 125 AD and still has the world's largest unreinforced concrete dome.", metadata={"topic": "architecture"}),
    Document(page_content="Gothic cathedrals use flying buttresses to transfer roof weight outward, enabling tall stained glass windows.", metadata={"topic": "architecture"}),
    Document(page_content="The Bauhaus movement combined fine arts and functional design, influencing modern architecture and typography.", metadata={"topic": "architecture"}),
    Document(page_content="Compound interest calculates returns on both the initial principal and previously earned interest.", metadata={"topic": "finance"}),
    Document(page_content="A stock represents partial ownership in a company and entitles the holder to a share of its profits.", metadata={"topic": "finance"}),
    Document(page_content="Diversification reduces portfolio risk by spreading investments across different asset classes.", metadata={"topic": "finance"}),
    Document(page_content="Shakespeare wrote 37 plays and 154 sonnets, exploring themes of power, love, and betrayal.", metadata={"topic": "literature"}),
    Document(page_content="The novel Don Quixote by Cervantes, published in 1605, is often considered the first modern novel.", metadata={"topic": "literature"}),
]

### BM25 Retrieval

This cell initializes a `BM25Retriever`, which is an efficient method for semantic search that relies purely on term frequency and inverse document frequency (TF-IDF) rather than dense vector embeddings. It builds an inverted index from the provided documents (`docs`) to enable fast, token-overlap scoring.


In [10]:
# BM25Retriever builds an inverted index from raw document text
# No embedding model is involved — scoring is purely based on token overlap
retriever = BM25Retriever.from_documents(docs, k=2) # Initialize the retriever using the documents and set k=2 to retrieve top 2 results


### Code Explanation

This cell demonstrates the retrieval process using a specific query designed for exact keyword matching. It uses the `retriever` object (which is configured with BM25) to fetch documents that contain the keywords 'antibiotic' and 'bacterial', showcasing BM25's strength in direct term matching.


In [12]:
# Query 1: exact keyword match — BM25 excels here
# The words "antibiotic" and "bacterial" appear directly in doc 1

query1 = "antibiotic bacterial infection treatment and killing pathogens"
results1 = retriever.invoke(query1)

print(f"Query: '{query1}'")
for i, doc in enumerate(results1, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
print()

Query: 'antibiotic bacterial infection treatment and killing pathogens'
  [1] topic=medicine: Antibiotics inhibit bacterial cell wall synthesis or protein production to stop infection.
  [2] topic=literature: Shakespeare wrote 37 plays and 154 sonnets, exploring themes of power, love, and betrayal.



### Code Explanation

This cell demonstrates the retrieval process using a specific, multi-faceted query related to finance. It uses the `retriever` object (likely an instance of a vector store or search engine) to find documents that contain keywords like 'compound interest', 'simple interest', and 'stakes' across different topics, simulating a real-world knowledge lookup.


In [17]:
# Query 2: keyword match across a different topic
# Words like "compound", "interest", and "returns" are present in finance docs

query2 = "compound interest vs Simple interest and what gives a person partial stakes in a company"
results2 = retriever.invoke(query2)
print(f"Query: '{query2}'")

for i, doc in enumerate(results2, 1):
    # Print the document number, its metadata topic, and the retrieved content.
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
print()


Query: 'compound interest vs Simple interest and what gives a person partial stakes in a company'
  [1] topic=finance: A stock represents partial ownership in a company and entitles the holder to a share of its profits.
  [2] topic=finance: Compound interest calculates returns on both the initial principal and previously earned interest.



### Code Explanation

This cell demonstrates a critical failure case for BM25: semantic queries that lack direct keyword overlap with the source documents. By querying for 'light and with windows,' which describes a concept rather than containing specific keywords, we test if the retriever can find semantically relevant information (like the Gothic cathedral) even when the exact words are missing.


In [20]:
# Query 3: semantic query with no keyword overlap — BM25 fails here
# None of the docs contain the words "light", "airy", or "tall windows" together
# The Gothic cathedral doc (doc 6) would be the ideal answer, but BM25 likely misses it

query3 = "a structure that feels light and with windows"

results3 = retriever.invoke(query3)
print(f"Query: '{query3}'")
for i, doc in enumerate(results3, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

Query: 'a structure that feels light and with windows'
  [1] topic=finance: A stock represents partial ownership in a company and entitles the holder to a share of its profits.
  [2] topic=literature: Shakespeare wrote 37 plays and 154 sonnets, exploring themes of power, love, and betrayal.
